# Dual-Window OCT to Full-Band OCT U-Net

This notebook trains a residual U-Net to predict the high-resolution full-band OCT B-scan from two dual-window OCT inputs:

- channel 0: short-wavelength dual-window B-scan
- channel 1: long-wavelength dual-window B-scan
- target: full-band OCT B-scan

Run the reconstruction/cache cell once. After the cache is saved, you can edit and rerun the model/training cells without reconstructing the OCT data again.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# If this import fails, install PyTorch in this Python environment first:
# python -m pip install torch torchvision
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    from oct_hsv_python_stage1.dual_window import (
        amplitude_to_db,
        reconstruct_training_pair_fast_cpu,
        reconstruct_training_pair_fast_gpu,
    )
except ModuleNotFoundError:
    # Use this fallback if you renamed oct_hsv_python_stage1 -> functions.
    from functions.dual_window import (
        amplitude_to_db,
        reconstruct_training_pair_fast_cpu,
        reconstruct_training_pair_fast_gpu,
    )

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1. Paths and Reconstruction Settings

Set these once for your dataset. `MAX_BSCANS` is useful for quick tests; use `None` for all B-scans in `test_data`.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "test_data"
PIXEL_MAP_PATH = DATA_DIR / "OCT_pixelMap.mat"
OUTPUT_DIR = PROJECT_DIR / "output_training"
CACHE_PATH = OUTPUT_DIR / "dual_window_training_cache.npz"

S1_GLOB = "*S1*.tif*"
S2_GLOB = "*S2*.tif*"

# Use None for all B-scans. Use a small number, e.g. 20, for quick debugging.
MAX_BSCANS = None

# Reconstruction settings
C2 = 38.02
C3 = -54.63
BROAD_WIDTH_FRACTION = 0.25
NARROW_WIDTH_FRACTION = 0.05
CENTER_SEPARATION_FRACTION = 0.75

# Set True only when you want to regenerate the cache.
REBUILD_CACHE = False

# Use CuPy/CUDA for OCT reconstruction when available.
USE_GPU_RECONSTRUCTION = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir:", DATA_DIR)
print("Cache path:", CACHE_PATH)

## 2. Helper Functions for TIFF Pair Discovery and Reconstruction Cache

This is the slow part. Once `CACHE_PATH` exists, later notebook runs should load it instead of reconstructing all B-scans.

In [ ]:
def find_tiff_pairs(data_dir: Path, s1_glob: str, s2_glob: str) -> tuple[list[Path], list[Path]]:
    s1_paths = sorted(data_dir.glob(s1_glob))
    s2_paths = sorted(data_dir.glob(s2_glob))
    if not s1_paths:
        raise FileNotFoundError(f"No S1 files found with {data_dir / s1_glob}")
    if len(s1_paths) != len(s2_paths):
        raise ValueError(f"Found {len(s1_paths)} S1 files and {len(s2_paths)} S2 files.")
    return s1_paths, s2_paths


def reconstruct_or_load_cache() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if CACHE_PATH.exists() and not REBUILD_CACHE:
        print(f"Loading cached training arrays: {CACHE_PATH}")
        with np.load(CACHE_PATH) as data:
            full = data["full"]
            short = data["short"]
            long = data["long"]
        print("Loaded shapes:", full.shape, short.shape, long.shape)
        return full, short, long

    s1_paths, s2_paths = find_tiff_pairs(DATA_DIR, S1_GLOB, S2_GLOB)
    if MAX_BSCANS is not None:
        s1_paths = s1_paths[:MAX_BSCANS]
        s2_paths = s2_paths[:MAX_BSCANS]

    full_slices = []
    short_slices = []
    long_slices = []

    backend = "GPU/CuPy" if USE_GPU_RECONSTRUCTION else "CPU/NumPy"
    print(f"Reconstructing {len(s1_paths)} B-scans from {DATA_DIR} with {backend}")
    for index, (s1_path, s2_path) in enumerate(zip(s1_paths, s2_paths), start=1):
        reconstruct_fn = (
            reconstruct_training_pair_fast_gpu
            if USE_GPU_RECONSTRUCTION
            else reconstruct_training_pair_fast_cpu
        )
        full, short, long = reconstruct_fn(
            s1_path,
            s2_path,
            PIXEL_MAP_PATH,
            c2=C2,
            c3=C3,
            broad_width_fraction=BROAD_WIDTH_FRACTION,
            narrow_width_fraction=NARROW_WIDTH_FRACTION,
            center_separation_fraction=CENTER_SEPARATION_FRACTION,
        )
        full_slices.append(full.astype(np.float32, copy=False))
        short_slices.append(short.astype(np.float32, copy=False))
        long_slices.append(long.astype(np.float32, copy=False))
        print(f"  reconstructed {index}/{len(s1_paths)}: {s1_path.name}")

    full = np.stack(full_slices, axis=0)
    short = np.stack(short_slices, axis=0)
    long = np.stack(long_slices, axis=0)

    metadata = {
        "data_dir": str(DATA_DIR),
        "pixel_map": str(PIXEL_MAP_PATH),
        "max_bscans": MAX_BSCANS,
        "broad_width_fraction": BROAD_WIDTH_FRACTION,
        "narrow_width_fraction": NARROW_WIDTH_FRACTION,
        "center_separation_fraction": CENTER_SEPARATION_FRACTION,
        "c2": C2,
        "c3": C3,
    }
    np.savez_compressed(
        CACHE_PATH,
        full=full,
        short=short,
        long=long,
        metadata=json.dumps(metadata),
    )
    print(f"Saved cache: {CACHE_PATH}")
    print("Saved shapes:", full.shape, short.shape, long.shape)
    return full, short, long

## 3. Run or Load Reconstruction Cache

Run this cell once. If the cache already exists and `REBUILD_CACHE = False`, this should be fast.

In [ ]:
full_amp, short_amp, long_amp = reconstruct_or_load_cache()

## 4. Normalize Data and Preview Training Inputs

The network trains on dB-scaled, clipped images in `[0, 1]`. The normalization limits are based on the full-band target volume.

In [ ]:
def normalize_volumes(
    full_amp: np.ndarray,
    short_amp: np.ndarray,
    long_amp: np.ndarray,
    low_percentile: float = 1.0,
    high_percentile: float = 99.8,
) -> tuple[np.ndarray, np.ndarray, float, float]:
    full_db = amplitude_to_db(full_amp).astype(np.float32)
    short_db = amplitude_to_db(short_amp).astype(np.float32)
    long_db = amplitude_to_db(long_amp).astype(np.float32)
    low, high = np.percentile(full_db, [low_percentile, high_percentile])

    def scale(x: np.ndarray) -> np.ndarray:
        return np.clip((x - low) / max(high - low, 1e-6), 0.0, 1.0).astype(np.float32)

    inputs = np.stack((scale(short_db), scale(long_db)), axis=1)  # (B, 2, depth, width)
    targets = scale(full_db)[:, None, :, :]                       # (B, 1, depth, width)
    return inputs, targets, float(low), float(high)


inputs, targets, DB_LOW, DB_HIGH = normalize_volumes(full_amp, short_amp, long_amp)
print("inputs:", inputs.shape)
print("targets:", targets.shape)
print(f"dB normalization range: {DB_LOW:.2f} to {DB_HIGH:.2f}")

In [ ]:
preview_index = inputs.shape[0] // 2
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, image, title in zip(
    axes,
    [targets[preview_index, 0], inputs[preview_index, 0], inputs[preview_index, 1]],
    ["Full-band target", "Short DW input", "Long DW input"],
):
    im = ax.imshow(image, cmap="gray", aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("A-line")
    ax.set_ylabel("Depth pixel")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

## 5. Dataset and Train/Validation Split

For this starter version, the final fraction of B-scans is used as validation. Once you add more volumes, split by volume/sample rather than randomly splitting adjacent B-scans.

In [ ]:
class BscanDataset(Dataset):
    def __init__(self, inputs: np.ndarray, targets: np.ndarray, indices: np.ndarray):
        self.inputs = inputs
        self.targets = targets
        self.indices = indices.astype(np.int64)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, item: int) -> tuple[torch.Tensor, torch.Tensor]:
        index = self.indices[item]
        return torch.from_numpy(self.inputs[index]), torch.from_numpy(self.targets[index])


def make_train_val_indices(n_slices: int, val_fraction: float) -> tuple[np.ndarray, np.ndarray]:
    indices = np.arange(n_slices)
    if n_slices < 5 or val_fraction <= 0:
        return indices, indices
    val_count = max(1, int(round(n_slices * val_fraction)))
    val_indices = indices[-val_count:]
    train_indices = indices[:-val_count]
    if train_indices.size == 0:
        train_indices = indices
    return train_indices, val_indices


VAL_FRACTION = 0.15
BATCH_SIZE = 1

train_indices, val_indices = make_train_val_indices(inputs.shape[0], VAL_FRACTION)
train_loader = DataLoader(BscanDataset(inputs, targets, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(BscanDataset(inputs, targets, val_indices), batch_size=1, shuffle=False, num_workers=0)

print(f"Training B-scans: {len(train_indices)}")
print(f"Validation B-scans: {len(val_indices)}")

## 6. Residual U-Net Model

This follows the paper-style residual U-Net idea: strided convolution for downsampling, pixel shuffle for upsampling, residual convolution blocks, and alpha-scaled residual additions.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )
        self.alpha = nn.Parameter(torch.tensor(0.1, dtype=torch.float32))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.alpha * self.block(x)


class DualWindowUNet(nn.Module):
    def __init__(self, in_channels: int = 2, out_channels: int = 1, base: int = 32):
        super().__init__()
        self.in_conv = nn.Conv2d(in_channels, base, kernel_size=3, padding=1)

        self.enc1 = nn.Sequential(*[ResidualBlock(base) for _ in range(4)])
        self.down1 = nn.Conv2d(base, base * 2, kernel_size=3, stride=2, padding=1)

        self.enc2 = nn.Sequential(*[ResidualBlock(base * 2) for _ in range(3)])
        self.down2 = nn.Conv2d(base * 2, base * 4, kernel_size=3, stride=2, padding=1)

        self.bottleneck = nn.Sequential(*[ResidualBlock(base * 4) for _ in range(4)])

        self.up1 = nn.Sequential(
            nn.Conv2d(base * 4, base * 2 * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
        )
        self.dec1 = nn.Sequential(*[ResidualBlock(base * 2) for _ in range(3)])

        self.up2 = nn.Sequential(
            nn.Conv2d(base * 2, base * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
        )
        self.dec2 = nn.Sequential(*[ResidualBlock(base) for _ in range(3)])

        self.out_conv = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        height, width = x.shape[-2:]
        pad_h = (4 - height % 4) % 4
        pad_w = (4 - width % 4) % 4
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")

        e1 = self.enc1(self.in_conv(x))
        e2 = self.enc2(self.down1(e1))
        bottleneck = self.bottleneck(self.down2(e2))

        d1 = self.dec1(self.up1(bottleneck) + e2)
        d2 = self.dec2(self.up2(d1) + e1)
        out = self.out_conv(d2)
        return out[..., :height, :width]

## 7. Training Setup

You can edit these values freely without rerunning reconstruction.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_CHANNELS = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 100
SAVE_EVERY = 10

model = DualWindowUNet(base=BASE_CHANNELS).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.L1Loss()

print("Device:", DEVICE)
print("Parameter count:", sum(p.numel() for p in model.parameters()))

## 8. Prediction Panel Helper

This saves one row with: full-band target, short DW input, long DW input, DL reconstruction.

In [ ]:
def save_prediction_panel(model: nn.Module, index: int, epoch: int) -> Path:
    model.eval()
    with torch.no_grad():
        x = torch.from_numpy(inputs[index:index + 1]).to(DEVICE)
        pred = model(x).cpu().numpy()[0, 0]

    images = [
        (targets[index, 0], "Full-band target"),
        (inputs[index, 0], "Short DW input"),
        (inputs[index, 1], "Long DW input"),
        (pred, "DL reconstruction"),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), constrained_layout=True)
    for ax, (image, title) in zip(axes, images):
        im = ax.imshow(image, cmap="gray", aspect="auto", vmin=0, vmax=1)
        ax.set_title(title)
        ax.set_xlabel("A-line")
        ax.set_ylabel("Depth pixel")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    figure_path = OUTPUT_DIR / f"epoch_{epoch:04d}_bscan_{index:04d}.png"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved prediction panel: {figure_path}")
    return figure_path

## 9. Train

This cell does not reconstruct OCT data. It uses the arrays already loaded in memory from the cache cell.

In [ ]:
history = {"epoch": [], "train_l1": [], "val_l1": []}
panel_index = int(val_indices[len(val_indices) // 2])

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for x, y in train_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        train_losses.append(float(loss.detach().cpu()))

    model.eval()
    val_losses = []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            val_losses.append(float(criterion(model(x), y).cpu()))

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses)) if val_losses else float("nan")
    history["epoch"].append(epoch)
    history["train_l1"].append(train_loss)
    history["val_l1"].append(val_loss)
    print(f"Epoch {epoch:04d}/{EPOCHS}: train L1={train_loss:.5f}, val L1={val_loss:.5f}")

    if epoch % SAVE_EVERY == 0 or epoch == 1 or epoch == EPOCHS:
        save_prediction_panel(model, panel_index, epoch)
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "db_low": DB_LOW,
                "db_high": DB_HIGH,
                "history": history,
            },
            OUTPUT_DIR / "latest_model.pt",
        )

## 10. Plot Training History

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.plot(history["epoch"], history["train_l1"], label="train L1")
ax.plot(history["epoch"], history["val_l1"], label="val L1")
ax.set_xlabel("Epoch")
ax.set_ylabel("L1 loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()